# Entrenament del Truc (etapa `cartes`) a Google Colab

**Factors importants abans d'executar:**

1. **Cal haver pujat els canvis locals a GitHub** (`git push`). Aquest notebook clona el repo des d'`origin`, no des del teu disc — si hi ha canvis sense pujar, Colab no els veurà. **Si el repositori és privat**, `git clone` fallarà sense credencials — més avall hi ha dues opcions: un token de GitHub, o pujar el projecte com a zip.
2. **Tria l'entorn d'execució CPU, no GPU**: Entorn d'execució → Canvia el tipus d'entorn d'execució → "Cap acceleració". Aquesta feina és de CPU (l'entorn del joc, no la xarxa), la GPU no ajuda i només gasta la quota limitada.
3. **Colab gratuït es desconnecta** (~90 min d'inactivitat, sessió màxima ~12h). Els checkpoints es guarden a Google Drive perquè sobrevisquin la desconnexió, i aquest notebook detecta sol si ja n'hi ha per continuar (`--resume_from`) en lloc de començar de zero cada vegada. **Quan es desconnecti, torna a obrir el notebook i executa totes les cel·les de nou.**
4. **La velocitat depèn totalment de la màquina que et toqui** — comprova-ho sempre amb la cel·la "Components d'aquest servidor" de sota abans de treure conclusions. No assumeixis que serà més lenta (ni més ràpida) que en local sense mirar-ho.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Components d'aquest servidor de Colab

Útil per saber què t'ha tocat (CPU/RAM/disc, i confirmar que NO hi ha GPU si has triat "Cap acceleració").

In [ ]:
import os
import platform

print("=== CPU ===")
print(f"Nuclis lògics (os.cpu_count): {os.cpu_count()}")
!nproc --all
!cat /proc/cpuinfo | grep "model name" | head -1

print("\n=== RAM ===")
!free -h

print("\n=== Disc ===")
!df -h /content /content/drive 2>/dev/null

print("\n=== GPU (hauria de sortir buit/error si has triat entorn CPU) ===")
!nvidia-smi 2>/dev/null || echo "Cap GPU assignada"

print("\n=== Versions ===")
print("Python:", platform.python_version())

## Configuració

`REPO_URL` ha de ser accessible (públic, o privat amb credencials configurades a Colab). `DRIVE_DIR` és on es clona el projecte dins de Google Drive (persistent entre sessions, així no cal re-clonar cada cop).

In [ ]:
REPO_URL = "https://github.com/JoFeF08/TFG-truc.git"
DRIVE_DIR = "/content/drive/MyDrive/tfg-truc"
BRANCH = "master"

In [ ]:
from getpass import getpass

# Nomes cal si el repositori es PRIVAT. Deixa-ho buit (Enter) si es public
# o si faras servir l'opcio de pujar un zip mes avall.
GITHUB_TOKEN = getpass("Token de GitHub (opcional, Enter per ometre'l): ")

In [ ]:
import os

clone_url = REPO_URL
if GITHUB_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if not os.path.exists(f"{DRIVE_DIR}/.git"):
    !git clone --branch {BRANCH} "{clone_url}" "{DRIVE_DIR}"
else:
    %cd {DRIVE_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

if os.path.exists(f"{DRIVE_DIR}/.git"):
    %cd {DRIVE_DIR}
    !git log -1 --oneline
else:
    print("\n⚠️ No s'ha pogut clonar (repositori privat sense token vàlid?).")
    print("Fes servir la cel·la 'Alternativa: pujar un zip' de sota en lloc d'aquesta.")

## Alternativa: pujar el projecte com a zip

Només si la cel·la de dalt ha fallat (repositori privat, sense token). En local, comprimeix el **contingut** de la carpeta del projecte (no la carpeta en si) — exclou `.venv/`, `.git/`, i qualsevol `registres_*`/`comparativa_ab` amb pesos grans, perquè el zip pesi poc i la pujada sigui ràpida. Executa només aquesta cel·la (no cal la de `git clone`).

In [ ]:
import os
import shutil
from google.colab import files

os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(f"{DRIVE_DIR}/RL"):
    print("Selecciona el .zip del projecte...")
    pujat = files.upload()
    nom_zip = list(pujat.keys())[0]
    shutil.unpack_archive(nom_zip, DRIVE_DIR)

    # Si el zip contenia una unica carpeta arrel (p.ex. "TFG-truc/"), en puja
    # el contingut un nivell perque quedi directament dins de DRIVE_DIR.
    contingut = [c for c in os.listdir(DRIVE_DIR) if c != nom_zip]
    if len(contingut) == 1 and os.path.isdir(os.path.join(DRIVE_DIR, contingut[0])):
        arrel = os.path.join(DRIVE_DIR, contingut[0])
        for item in os.listdir(arrel):
            shutil.move(os.path.join(arrel, item), DRIVE_DIR)
        os.rmdir(arrel)

    print(f"Descomprimit a {DRIVE_DIR}")
else:
    print("Ja hi ha un projecte a DRIVE_DIR -- no cal pujar-lo de nou.")

%cd {DRIVE_DIR}

## Dependències

Torch en versió CPU (sense CUDA) — la feina no fa servir GPU, i la roda CUDA és molt més gran i lenta de descarregar.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q stable-baselines3 sb3-contrib gymnasium pettingzoo tqdm

In [ ]:
import os
n_cpus = os.cpu_count()
# Es deixen un parell de nuclis lliures per al proces principal (calcul del
# gradient), la resta per als subprocessos de l'entorn (SubprocVecEnv).
NUM_ENVS = max(1, n_cpus - 2)
print(f"CPUs disponibles a aquesta màquina de Colab: {n_cpus} -> --num_envs {NUM_ENVS}")

## Represa automàtica

Els checkpoints es guarden a `POOL_DIR`, dins de Google Drive (sobreviuen la desconnexió). Si ja n'hi ha algun, es reprèn l'entrenament en lloc de començar de zero.

In [ ]:
from pathlib import Path

POOL_DIR = f"{DRIVE_DIR}/RL/entrenament/registres_colab/pool"
Path(POOL_DIR).mkdir(parents=True, exist_ok=True)

checkpoints = sorted(
    Path(POOL_DIR).glob("*_steps.zip"),
    key=lambda p: int(p.stem.rsplit('_', 2)[-2]),
)
if checkpoints:
    print(f"Checkpoint previ trobat: {checkpoints[-1].name} -- es reprendrà des d'aquí.")
else:
    print("Cap checkpoint previ -- entrenament nou des de zero.")

## Entrenament

Mateixa configuració que en local (`--opponent fort` amb `AgentProbabilistic`, `--reward_scale 15 --ent_coef 0.01`), amb `--target_kl 0.03` afegit (l'entrenament local va mostrar `approx_kl`/`clip_fraction` sostingudament alts amb `reward_scale=15` -- `target_kl` fa que PPO aturi cada època d'actualització si el KL el supera, evitant la divergència).

In [ ]:
cmd = (
    f'python RL/entrenament/entrenament_sb3.py '
    f'--stage cartes --opponent fort '
    f'--reward_scale 15 --ent_coef 0.01 --target_kl 0.03 '
    f'--num_envs {NUM_ENVS} --total_timesteps 12000000 '
    f'--pool_dir "{POOL_DIR}"'
)
if checkpoints:
    cmd += f' --resume_from "{POOL_DIR}"'

print(cmd)
!{cmd}

## Si es desconnecta la sessió

Torna a obrir aquest notebook i executa totes les cel·les de dalt a baix (`Entorn d'execució` → `Executa-ho tot`). La cel·la de "Represa automàtica" detectarà el darrer checkpoint a `POOL_DIR` (a Google Drive, no es perd) i la cel·la d'entrenament hi continuarà amb `--resume_from` en lloc de començar de nou.

## Avaluació dels checkpoints (repartiments duplicats)

Fes-ho des d'una **segona pestanya/runtime de Colab** connectada a la mateixa Google Drive, per no interrompre l'entrenament que ja corre. Mateix mètode que en local (`RL/tools/avaluacio_duplicada.py`): repartiments duplicats, molt menys soroll que un win-rate senzill.

1. Comparativa completa del darrer checkpoint contra `random`, els 4 estils de `regles`, i `probabilistic` (el solver quasi-òptim).
2. Corba d'aprenentatge (`vs random`) sobre una mostra repartida de checkpoints — el "final" no sempre és el millor (ho vam veure al run local: el de 1.6M anava millor que el de 2M).

In [ ]:
from pathlib import Path

checkpoints = sorted(
    Path(POOL_DIR).glob("*_steps.zip"),
    key=lambda p: int(p.stem.rsplit('_', 2)[-2]),
)

if not checkpoints:
    print("Encara no hi ha cap checkpoint.")
else:
    darrer = checkpoints[-1]
    print(f"=== Comparativa completa del darrer checkpoint: {darrer.name} ===\n")
    for rival in ["random", "conservador", "equilibrat", "agressiu", "farol", "probabilistic"]:
        !python -m RL.tools.avaluacio_duplicada --model "{darrer}" --rival {rival} --n_repartiments 150

    # Corba d'aprenentatge: mostra fins a 6 checkpoints repartits al llarg de
    # l'entrenament (el "final" no sempre es el millor, ja ho vam veure en local).
    if len(checkpoints) > 1:
        mostra = checkpoints[:: max(1, len(checkpoints) // 6)]
        if checkpoints[-1] not in mostra:
            mostra.append(checkpoints[-1])
        print(f"\n=== Corba d'aprenentatge (vs random), {len(mostra)} checkpoints ===\n")
        for ck in mostra:
            !python -m RL.tools.avaluacio_duplicada --model "{ck}" --rival random --n_repartiments 100